# mercymek · FastGS for the 3D moments

Trains one [FastGS](https://github.com/fastgs/FastGS) scene per 3D moment (MIT; 3DGS-based, CUDA only) and returns the
Gaussians for `scripts/fastgs_import.py`.

1. Upload `data/fastgs/mercy_fastgs.zip` (from `python scripts/fastgs_export.py`) as a **private** Kaggle dataset and
   add it to this notebook.
2. Settings: Accelerator **GPU T4** (FastGS needs compute capability ≥ 7.0, so not the P100), Internet **on**.
3. Run all. Download `fastgs_out.zip` from the output and run `python scripts/fastgs_import.py fastgs_out.zip`.

What is patched in FastGS (inherited from the original 3DGS code): the alpha channel of RGBA images was never loaded
(`shape[1] == 4` instead of `shape[0]`), and alpha only blackened the target instead of removing those pixels from the
loss. Here alpha is the loss mask: the photometric loss and FastGS's multi-view error maps ignore masked pixels
(moving vehicles in every frame except the moment's own, the sky, the camera car's hood).

In [ ]:
ITERS = 10_000          # FastGS reaches most of its quality early; ~3-5 min per moment on a T4
WIDTH = 1280            # training resolution (the export is 1280 wide)
import pathlib, json, os, subprocess, sys, time, shutil
DATA = next(p.parent for p in pathlib.Path('/kaggle/input').rglob('index.json'))
WORK = pathlib.Path('/kaggle/working')
moments = json.loads((DATA / 'index.json').read_text())
print(DATA, len(moments), 'moments')

In [ ]:
%%bash
set -e
cd /kaggle/working
[ -d FastGS ] || git clone --depth 1 https://github.com/fastgs/FastGS
nvidia-smi --query-gpu=name,compute_cap --format=csv
export TORCH_CUDA_ARCH_LIST="7.5"
pip -q install plyfile
cd FastGS
pip -q install --no-build-isolation ./submodules/diff-gaussian-rasterization_fastgs ./submodules/simple-knn ./submodules/fused-ssim

In [ ]:
import pathlib
R = pathlib.Path('/kaggle/working/FastGS')

def patch(f, old, new):
    p = R / f
    s = p.read_text()
    if new in s:
        return
    assert old in s, f'{f}: patch target not found (FastGS changed?)'
    p.write_text(s.replace(old, new, 1))

# 1. RGBA alpha is channel 0 of the tensor's shape, not 1 (3DGS bug): load the mask at all
patch('utils/camera_utils.py', 'if resized_image_rgb.shape[1] == 4:', 'if resized_image_rgb.shape[0] == 4:')
# 2. keep the mask on the camera
patch('scene/cameras.py',
      """        if gt_alpha_mask is not None:
            self.original_image *= gt_alpha_mask.to(self.data_device)
        else:
            self.original_image *= torch.ones((1, self.image_height, self.image_width), device=self.data_device)""",
      """        self.loss_mask = gt_alpha_mask.to(self.data_device) if gt_alpha_mask is not None else None
        if self.loss_mask is not None:
            self.original_image *= self.loss_mask""")
# 3. masked pixels give no photometric loss
patch('train.py',
      """        gt_image = viewpoint_cam.original_image.cuda()
        Ll1 = l1_loss(image, gt_image)""",
      """        gt_image = viewpoint_cam.original_image.cuda()
        if viewpoint_cam.loss_mask is not None:
            image = image * viewpoint_cam.loss_mask
        Ll1 = l1_loss(image, gt_image)""")
# 4. ... and no error in FastGS's multi-view densification / pruning maps
patch('utils/fast_utils.py',
      """        render_image = render_fastgs(my_viewpoint_cam, gaussians, pipe, bg, args.mult)["render"]
""",
      """        render_image = render_fastgs(my_viewpoint_cam, gaussians, pipe, bg, args.mult)["render"]
        if my_viewpoint_cam.loss_mask is not None:
            render_image = render_image * my_viewpoint_cam.loss_mask
""")
print('FastGS patched')

In [ ]:
results = []
for m in moments:
    name = f"{m['case']}/{m['moment']}"
    src = WORK / 'data' / name                     # COLMAP loading writes points3D.ply next to the input
    shutil.copytree(DATA / name, src, dirs_exist_ok=True)
    out = WORK / 'models' / name
    t0 = time.time()
    cmd = [sys.executable, 'train.py', '-s', str(src), '-i', 'images', '-m', str(out), '-r', str(WIDTH),
           '--iterations', str(ITERS), '--position_lr_max_steps', str(ITERS), '--densify_until_iter', str(ITERS // 2),
           '--densification_interval', '500', '--optimizer_type', 'default',
           '--save_iterations', str(ITERS), '--test_iterations', str(ITERS), '--checkpoint_iterations', str(ITERS + 1), '--quiet']
    r = subprocess.run(cmd, cwd=WORK / 'FastGS', capture_output=True, text=True)
    secs = round(time.time() - t0, 1)
    ply = out / 'point_cloud' / f'iteration_{ITERS}' / 'point_cloud.ply'
    if r.returncode or not ply.exists():
        print(name, 'FAILED', r.stdout[-1500:], r.stderr[-1500:])
        continue
    dst = WORK / 'fastgs_out' / name
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copy(ply, dst / 'point_cloud.ply')
    shutil.copy(DATA / name / 'moment.json', dst / 'moment.json')
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True).stdout.strip()
    log = {'iterations': ITERS, 'seconds': secs, 'width': WIDTH, 'gpu': gpu, 'mb': round(ply.stat().st_size / 1e6, 1)}
    (dst / 'train_log.json').write_text(json.dumps(log))
    results.append({'moment': name, **log})
    print(name, log)

In [ ]:
shutil.make_archive(str(WORK / 'fastgs_out'), 'zip', WORK / 'fastgs_out')
print(len(results), 'of', len(moments), 'moments trained ->', WORK / 'fastgs_out.zip')